# Notebook 2 — Retrieval for Pipeline A
**Owner:** A2. **Prereqs:** Notebook 1 §0–§4, §10, §11 completed by A1; §8 completed by A2.
**Time:** setup + relevance seeding ~15 min; retrieval after team call ~10 min.

**Reads:** `03_CORPUS/`, `04_GOLD/`, `02_SHARED_CODE/`
**Writes:** `10_PIPELINE_A/results_A/retrieval_results_A.csv`, `inverted_index_A.json`, `data/relevance.tsv`

---

## Ownership of retrieval in the team split

This notebook produces Pipeline A's retrieval evaluation only. B2 runs a parallel copy of this notebook for Pipeline B, writing to `20_PIPELINE_B/results_B/`. The A vs B comparison, Final Pipeline selection, and GUI live in a separate joint notebook that B2 owns on Day 4, once both pipelines' `retrieval_results_*.csv` files exist.

**Do not run §3 (retrieval) until `04_GOLD/relevance.tsv` has been hand-verified by all four teammates.** See the STOP block below §2 for why.

## §0 Setup — run every session

In [ ]:
!pip install -q nltk spacy tiktoken scikit-learn huggingface_hub pandas requests
!python -m spacy download en_core_web_sm --quiet

In [ ]:
import os, sys
from google.colab import drive
drive.mount('/content/drive')

BASE    = '/content/drive/MyDrive/NLP Assignment 1'
CODE    = f'{BASE}/02_SHARED_CODE'
DATA    = f'{BASE}/03_CORPUS'
GOLD    = f'{BASE}/04_GOLD'
RESULTS = f'{BASE}/10_PIPELINE_A/results_A'

for d in (DATA, GOLD, RESULTS): os.makedirs(d, exist_ok=True)
sys.path.insert(0, CODE)

# Same monkey-patch as Notebook 1: route vgir's file I/O into our Drive layout.
os.environ['VGIR_BASE'] = BASE
import vgir
def P(*parts):
    if not parts: return BASE
    head = parts[0]
    if head == 'data':    return os.path.join(DATA, *parts[1:])
    if head == 'results': return os.path.join(RESULTS, *parts[1:])
    if head == 'domain_lexicon.json': return f'{BASE}/01_FROZEN/domain_lexicon.json'
    return os.path.join(BASE, *parts)
vgir.P = P
def write_csv(rows, name, folder='results'):
    import pandas as pd
    target = RESULTS if folder == 'results' else os.path.join(BASE, folder)
    os.makedirs(target, exist_ok=True)
    df = pd.DataFrame(rows); df.to_csv(os.path.join(target, name), index=False); return df
vgir.write_csv = write_csv

print('BASE   :', BASE, '  exists=', os.path.isdir(BASE))
print('vgir   :', vgir.__file__)
print('corpus :', DATA, ' documents=', len([f for f in os.listdir(DATA) if f.startswith('D')]))

## §1 Rebuild the 12-query list (must match Notebook 1 exactly)

This block regenerates the same QUERIES table that Notebook 1 §1 produced, by re-running the auto-picker against the MIRACL topics. The picker is deterministic — same regex, same tie-break rule — so if `03_CORPUS/topics.miracl-*.tsv` hasn't changed, the 12 queries here match what was used to build the corpus.

**We also decorate queries with Boolean / phrase operators here.** The assignment requires the 12 queries to cover keyword, phrase, AND, OR and NOT types. The corpus was fetched using the raw questions (no operators), but retrieval evaluates the operator-decorated forms.

In [ ]:
import re
from collections import Counter, defaultdict

# --- Rebuild QUERIES exactly as Notebook 1 did ------------------------
def load_qrels(path):
    d = defaultdict(dict)
    with open(path) as f:
        for line in f:
            qid, _, docid, rel = line.split()
            d[qid][docid] = int(rel)
    return d
qrels = {}
for split in ('train','dev'):
    for k, v in load_qrels(f'{DATA}/qrels.miracl-v1.0-en-{split}.tsv').items():
        qrels[f'{split}:{k}'] = v

VG_RX = re.compile(
    r"video ?games?|game ?boy|nintendo|playstation|xbox|\bsega\b|genesis|atari|\bconsoles?\b|"
    r"pok[e\u00e9]mon|zelda|metroid|\bhalo\b|\bdoom\b|wolfenstein|counter-strike|warcraft|"
    r"final fantasy|metal gear|tomb raider|silent hill|resident evil|mortal kombat|"
    r"gears of war|superhot|kingdom hearts|assassin'?s? creed|dragon quest|"
    r"monkey island|rock band|guitar hero|last of us|grand theft auto|total war|"
    r"\bsnes\b|super nintendo|\brpg\b|\bfps\b|shooter|adventure game|"
    r"point and click|first-person|third-person|role-playing", re.I)

def load_topics(path):
    with open(path) as f:
        return [line.rstrip('\n').split('\t', 1) for line in f if '\t' in line]

candidates = []
for split in ('train','dev'):
    for qid, text in load_topics(f'{DATA}/topics.miracl-v1.0-en-{split}.tsv'):
        if not VG_RX.search(text): continue
        key = f'{split}:{qid.strip()}'
        pos = [d.split('#')[0] for d,r in qrels.get(key,{}).items() if r > 0]
        if len(pos) >= 2:
            candidates.append({'qid':key,'query':text.strip(),'n_pos':len(pos)})

def cluster(q):
    ql = q.lower()
    if any(w in ql for w in ['game boy','gameboy','nintendo','sega','xbox','playstation',
                              'atari','console','snes','super nintendo','genesis']): return 'consoles'
    if any(w in ql for w in ['shooter','rpg','role-playing','adventure','strategy','fps',
                              'first-person','third-person','point and click']): return 'genre'
    if any(w in ql for w in ['sold','sales','revenue','copies','units','million','billion']): return 'sales'
    if 'first' in ql or 'earliest' in ql or 'oldest' in ql: return 'firsts'
    return 'franchises'
for c in candidates: c['cluster'] = cluster(c['query'])

picked, per_cluster = [], Counter()
for c in sorted(candidates, key=lambda x: (-x['n_pos'], x['qid'])):
    if per_cluster[c['cluster']] >= 4: continue
    picked.append(c); per_cluster[c['cluster']] += 1
    if len(picked) == 12: break
if len(picked) < 12:
    for c in sorted(candidates, key=lambda x: -x['n_pos']):
        if c not in picked: picked.append(c)
        if len(picked) == 12: break

# --- Decorate with Boolean / phrase forms for the retrieval evaluation ---
# Assignment §5 requires coverage of: keyword, phrase, AND, OR, NOT.
# We take the auto-picked raw question and derive an operator form based on cluster + rank.
STOPS_LITE = set('a an the of and or to in is was who what when where how did does '
                 'is are for on by with from as it its this that many first last '
                 'been have has main make made version'.split())
def content_terms(q):
    return [w for w in re.findall(r"[A-Za-z][A-Za-z\-']+", q) if w.lower() not in STOPS_LITE]

# Assign 12 slots to query types deterministically:
# 3 keyword, 3 phrase, 2 AND, 2 OR, 2 NOT.
type_plan = ['keyword','phrase','and','or','not','keyword','phrase','and','or','not','keyword','phrase']

def to_form(raw, typ):
    words = content_terms(raw)
    if not words: return raw, 'keyword'
    if typ == 'keyword':
        return words[0], 'keyword'
    if typ == 'phrase':
        n = min(3, len(words))
        return f'"{" ".join(words[:n])}"', 'phrase'
    if typ == 'and' and len(words) >= 2:
        return f'{words[0]} AND {words[1]}', 'and'
    if typ == 'or' and len(words) >= 2:
        return f'{words[0]} OR {words[1]}', 'or'
    if typ == 'not' and len(words) >= 2:
        return f'{words[0]} AND NOT {words[1]}', 'not'
    return words[0], 'keyword'

QUERIES = []
for i, c in enumerate(picked):
    form, typ = to_form(c['query'], type_plan[i])
    QUERIES.append((c['qid'], f'Q{i+1:02d}', form, typ, c['cluster'], c['query']))

import pandas as pd
df = pd.DataFrame(QUERIES, columns=['miracl_qid','our_qid','query','type','cluster','original_question'])
print(f'query-type distribution: {Counter(t for _,_,_,t,_,_ in QUERIES)}\n')
df

## §2 Seed `04_GOLD/relevance.tsv` from MIRACL, then STOP

This cell writes an initial `relevance.tsv` where every (query, document) pair is marked `0` (not relevant), except pairs where the document's source Wikipedia article was one of that query's positive MIRACL judgments — those are marked `1`.

This seed is a starting point, not the truth. Many documents in our 50-document corpus are relevant to queries other than the one they were fetched for (e.g. the Nintendo article is relevant to Zelda queries, Game Boy queries, and "first video game" queries), but the seed won't know that.

**After running this cell, STOP.** Do not proceed to §3 until all four teammates have hand-verified every row of `relevance.tsv` in a joint session. Once verified, `LOCKED_ON.txt` gets written and the file is treated as immutable.

Why stop here: if you run §3 before judgments are complete, you get low-recall retrieval numbers based on incomplete relevance, which cannot be compared with Pipeline B's later numbers unless B2 waits for the same broken seed. Do it right: seed, verify together, then run retrieval.

In [ ]:
import pandas as pd
REL = f'{GOLD}/relevance.tsv'
LOCK = f'{GOLD}/relevance_LOCKED_ON.txt'

manifest = pd.read_csv(f'{DATA}/manifest.csv')

if os.path.isfile(LOCK):
    print(f'relevance.tsv was locked on {open(LOCK).read().strip()}. Skipping seed.')
elif os.path.isfile(REL):
    n = sum(1 for _ in open(REL)) - 1
    print(f'{REL} already exists ({n} rows). Not overwriting. '
          f'If the four-teammate verification is complete, create {LOCK} with today\'s date.')
else:
    rows = []
    for _, ourq, _, _, _, _ in QUERIES:
        for did in manifest.doc_id:
            r = manifest[manifest.doc_id == did].iloc[0]
            rel = 1 if (r.source_our_qid == ourq and r.role == 'positive') else 0
            rows.append((ourq, did, rel))
    pd.DataFrame(rows, columns=['our_qid','doc_id','rel']).to_csv(REL, sep='\t', index=False)
    print(f'Seeded {REL} with {len(rows)} rows.')

print('\n' + '=' * 60)
print(' STOP. Do the four-teammate verification call now.')
print(' Every row of relevance.tsv must be reviewed. Change 0->1 where')
print(' a document is actually relevant to the query. Nobody edits')
print(' after the call. Then write today\'s date to:')
print(f' {LOCK}')
print(' Only then run §3.')
print('=' * 60)

## §3 Build the Pipeline A index and evaluate the 12 queries

This block:
1. Builds the positional inverted index using Pipeline A (from `vgir.PIPELINES['A']`).
2. Saves the index to `inverted_index_A.json`.
3. Runs each of the 12 queries, records cold and warm timings.
4. Computes Precision, Recall, F1, P@5, P@10, R@5, R@10 against `relevance.tsv`.
5. Writes `retrieval_results_A.csv`.

**Guard:** the first cell refuses to proceed unless `relevance_LOCKED_ON.txt` exists. Create that file after the team verification call.

In [ ]:
# Guard: don't run retrieval on unverified judgments.
REL = f'{GOLD}/relevance.tsv'
LOCK = f'{GOLD}/relevance_LOCKED_ON.txt'
assert os.path.isfile(REL), 'relevance.tsv not found; run §2 first'
assert os.path.isfile(LOCK), (
    'relevance.tsv is not LOCKED yet. Complete the four-teammate verification '
    'call, then write today\'s date to ' + LOCK + '. Refusing to run retrieval on '
    'unverified judgments.')
print(f'relevance.tsv locked on {open(LOCK).read().strip()}. Proceeding.')

In [ ]:
# Reload vgir in case §0's monkey-patches got clobbered.
import importlib; importlib.reload(vgir); vgir.P = P; vgir.write_csv = write_csv

docs = vgir.load_corpus(DATA)
print(f'{len(docs)} documents loaded.')

ixA = vgir.PositionalIndex(vgir.pipeline_A, 'A').build(docs)
print(f'Pipeline A index: vocab={ixA.vocab_size()}, tokens={ixA.token_count()}')

import json
with open(f'{RESULTS}/inverted_index_A.json', 'w') as f:
    json.dump({'A': ixA.to_json()}, f)
print(f'wrote {RESULTS}/inverted_index_A.json')

In [ ]:
# Load relevance judgments
import pandas as pd
rel = pd.read_csv(REL, sep='\t')
REL_SET = {(r.our_qid, r.doc_id) for r in rel.itertuples() if r.rel > 0}
print(f'{len(REL_SET)} positive (query, doc) pairs across {rel.our_qid.nunique()} queries')

In [ ]:
# Run all 12 queries; record ranked results, timings, and metrics.
K_VALUES = [5, 10]   # locked in PREREGISTRATION.md §6

qeA = vgir.QueryEngine(ixA)
rows = []
for miracl_qid, ourq, query, qtype, cluster, original in QUERIES:
    ranked, positive, cold, warm = qeA.timed_search(query, repeats=5)
    gold = {d for (q, d) in REL_SET if q == ourq}
    P_, R_, F_ = vgir.set_metrics(ranked, gold)
    pk = {k: vgir.p_at_k(ranked, gold, k) for k in K_VALUES}
    rk = {k: vgir.r_at_k(ranked, gold, k) for k in K_VALUES}
    rows.append({
        'pipeline':'A','our_qid':ourq,'query':query,'type':qtype,'cluster':cluster,
        'n_retrieved':len(ranked),'n_relevant':len(gold),
        'P':round(P_,3),'R':round(R_,3),'F1':round(F_,3),
        **{f'P@{k}':round(v,3) for k,v in pk.items()},
        **{f'R@{k}':round(v,3) for k,v in rk.items()},
        'cold_ms':round(cold,2),'warm_ms':round(warm,2),
        'retrieved_top10':' '.join(ranked[:10])
    })

df = vgir.write_csv(rows, 'retrieval_results_A.csv')
print('\nMacro averages (Pipeline A):')
print(df[['P','R','F1','P@5','P@10','R@5','R@10','warm_ms']].mean().round(3).to_string())
print('\nPer-query:')
print(df[['our_qid','type','n_retrieved','n_relevant','P','R','F1']].to_string(index=False))

---
## Outputs written to `10_PIPELINE_A/results_A/`

- `inverted_index_A.json`
- `retrieval_results_A.csv`

B2 runs a parallel copy of this notebook for Pipeline B and writes `retrieval_results_B.csv` and `inverted_index_B.json` to `20_PIPELINE_B/results_B/`. On Day 4, B2 assembles the joint comparison notebook that reads both and produces `pipeline_comparison.csv`, `evaluation_results.csv`, and the GUI.

## Report drafting notes for A2

Once retrieval is done, A2 can start drafting Module 3 (Pipeline A's design justification) and Module 6 (Pipeline A's retrieval evaluation) in `40_REPORT/Report_draft.gdoc`. The Pipeline A vs B comparison prose waits until B2's numbers are in — writing the comparison before both sides are measured invites confirmation bias.